In [128]:
%load_ext autoreload
%autoreload 2
    
from dataclasses import dataclass
from tqdm.notebook import tqdm
from PIL import Image

import json
import glob
import numpy as np
import os
import pandas as pd
import pyrender
import re
import shutil
import trimesh
import voxeloo

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Helper block color definitions

In [96]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int
        
def to_flora_id(terrain_id):
    return (1 << 24) | (terrain_id & 0xffffff)

# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x323143ff),
    TerrainVoxel("bedrock", 6, 0x4d3f53ff),
    TerrainVoxel("birch_log", 12, 0xb9c8d2ff),
    TerrainVoxel("clay", 17, 0x755a56ff),
    TerrainVoxel("coal_ore", 19, 0x2f2d34ff),
    TerrainVoxel("cobblestone", 5, 0x706e76ff),
    TerrainVoxel("diamond_ore", 23, 0x36d9f1ff),
    TerrainVoxel("dirt", 2, 0x9b633cff),
    TerrainVoxel("gold_ore", 11, 0xded02cff),
    TerrainVoxel("granite", 13, 0xb2bcc7ff),
    TerrainVoxel("grass", 1, 0x30a144ff),
    TerrainVoxel("gravel", 36, 0xad8c64ff),
    TerrainVoxel("hay", 35, 0xcdb158ff),
    TerrainVoxel("limestone", 9, 0x8b7b70ff),
    TerrainVoxel("moss", 40, 0x247447ff),
    TerrainVoxel("neptunium_ore", 30, 0x394f57ff),
    TerrainVoxel("oak_log", 3, 0x774530ff),
    TerrainVoxel("pumpkin", 10, 0xc67a2bff),
    TerrainVoxel("quartzite", 7, 0x527d79ff),
    TerrainVoxel("rubber_log", 14, 0x603526ff),
    TerrainVoxel("silver_ore", 25, 0x7f9cbaff),
    TerrainVoxel("stone", 4, 0x828690ff),
    TerrainVoxel("snow", 37, 0xe5f6faff),

    # Crafted blocks
    TerrainVoxel("basalt_brick", 15, 0x323143ff),
    TerrainVoxel("basalt_carved", 41, 0x323143ff),
    TerrainVoxel("basalt_polished", 42, 0x323143ff),
    TerrainVoxel("basalt_shingles", 43, 0x323143ff),
    TerrainVoxel("birch_lumber", 16, 0xc29b44ff),
    TerrainVoxel("clay_brick", 18, 0x883b49ff),
    TerrainVoxel("clay_carved", 44, 0x883b49ff),
    TerrainVoxel("clay_polished", 45, 0x883b49ff),
    TerrainVoxel("clay_shingles", 46, 0x883b49ff),
    TerrainVoxel("cobblestone_brick", 20, 0x706e76ff),
    TerrainVoxel("cobblestone_carved", 47, 0x706e76ff),
    TerrainVoxel("cobblestone_polished", 48, 0x706e76ff),
    TerrainVoxel("cobblestone_shingles", 49, 0x706e76ff),
    TerrainVoxel("cotton_fabric", 38, 0xe9e8e9ff),
    TerrainVoxel("diamond", 21, 0xa6f6ffff),
    TerrainVoxel("gold", 24, 0xf7ca00ff),
    TerrainVoxel("granite_brick", 26, 0xb2bcc7ff),
    TerrainVoxel("granite_carved", 50, 0xb2bcc7ff),
    TerrainVoxel("granite_polished", 51, 0xb2bcc7ff),
    TerrainVoxel("granite_shingles", 52, 0xb2bcc7ff),
    TerrainVoxel("limestone_brick", 27, 0x8b7b70ff),
    TerrainVoxel("limestone_carved", 53, 0x8b7b70ff),
    TerrainVoxel("limestone_polished", 54, 0x8b7b70ff),
    TerrainVoxel("limestone_shingles", 55, 0x8b7b70ff),
    TerrainVoxel("mushroom_leather", 39, 0x7c5a45ff),
    TerrainVoxel("neptunium", 28, 0x224a3eff),
    TerrainVoxel("oak_lumber", 31, 0x8b572aff),
    TerrainVoxel("quartzite_brick", 32, 0x527d79ff),
    TerrainVoxel("quartzite_carved", 56, 0x527d79ff),
    TerrainVoxel("quartzite_polished", 57, 0x527d79ff),
    TerrainVoxel("quartzite_shingles", 58, 0x527d79ff),
    TerrainVoxel("rubber_lumber", 34, 0x764330ff),
    TerrainVoxel("silver", 33, 0xccd6dbff),
    TerrainVoxel("stone_brick", 29, 0x828690ff),
    TerrainVoxel("stone_carved", 59, 0x828690ff),
    TerrainVoxel("stone_polished", 60, 0x828690ff),
    TerrainVoxel("stone_shingles", 61, 0x828690ff),
    TerrainVoxel("template", 69, 0x990099ff),
    TerrainVoxel("wood_crate", 22, 0x8b572aff),

    # Flora
    TerrainVoxel("oak_leaf", to_flora_id(1), 0x125d37ff),
    TerrainVoxel("birch_leaf", to_flora_id(2), 0x4e6a1dff),
    TerrainVoxel("rubber_leaf", to_flora_id(3), 0x20575aff),
    TerrainVoxel("switch_grass", to_flora_id(4), 0x4bbb47ff),
    TerrainVoxel("azalea_flower", to_flora_id(5), 0xc86c8dff),
    TerrainVoxel("bell_flower", to_flora_id(6), 0x59a5d5ff),
    TerrainVoxel("dandelion_flower", to_flora_id(7), 0xd0c227ff),
    TerrainVoxel("daylily_flower", to_flora_id(8), 0xcd8216ff),
    TerrainVoxel("lilac_flower", to_flora_id(9), 0xa767f2ff),
    TerrainVoxel("rose_flower", to_flora_id(10), 0xcc374cff),
    TerrainVoxel("cotton_flower", to_flora_id(11), 0xe9e8e9ff),
    TerrainVoxel("hemp_bush", to_flora_id(12), 0x2b6230ff),
    TerrainVoxel("red_mushroom", to_flora_id(13), 0xb02f3aff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

terrain_color = np.zeros(max(tv.index for tv in terrain) + 1, dtype=np.uint32)
for tv in terrain:
    terrain_color[tv.index] = tv.color

In [97]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Load shard files

In [98]:
DUMP_DIR = "/Users/taylorgordon/data/dumps/2023_07_27_19_13_01"

In [99]:
def visualize_shard(shard):
    visualize_voxels(terrain_color[shard.array()])

In [100]:
def shard_id_from_path(path):
    return tuple(map(int, os.path.basename(path)[:-len(".diff.bin")].split("_")))

### Scan all shards and produce some stats on blocks.

In [114]:
def load_volume_u32(kind):
    ret = {}
    for path in tqdm(glob.glob(f"{DUMP_DIR}/*.{kind}.bin")):
        block = voxeloo.biomes.VolumeBlock_U32()
        with open(path, "rb") as f:
            block.compressed_loads(f.read())
        key = re.match(f"{DUMP_DIR}/(.*).{kind}.bin", path)[1]
        ret[key] = block
    return ret

In [115]:
def load_sparse_u32(kind):
    ret = {}
    for path in tqdm(glob.glob(f"{DUMP_DIR}/*.{kind}.bin")):
        block = voxeloo.biomes.SparseBlock_U32()
        with open(path, "rb") as f:
            block.compressed_loads(f.read())
        key = re.match(f"{DUMP_DIR}/(.*).{kind}.bin", path)[1]
        ret[key] = block
    return ret

In [116]:
def load_tensor_f64(kind):
    ret = {}
    for path in tqdm(glob.glob(f"{DUMP_DIR}/*.{kind}.bin")):
        block = voxeloo.tensors.Tensor_F64([32, 32, 32])
        with open(path, "rb") as f:
            block.load(f.read())
        key = re.match(f"{DUMP_DIR}/(.*).{kind}.bin", path)[1]
        ret[key] = block
    return ret

In [117]:
diffs = load_sparse_u32("diff")

  0%|          | 0/262144 [00:00<?, ?it/s]

In [118]:
placers = load_tensor_f64("placer")

  0%|          | 0/262144 [00:00<?, ?it/s]

In [139]:
placer_counts = {}

for key, placer in tqdm(placers.items()):
    diff = diffs[key]
    coords = np.stack(placer.array().nonzero(), axis=-1)
    for z, y, x in coords.tolist():
        id = int(placer.get(x, y, z))
        counts = placer_counts.setdefault(id, {})
        
        terrain_id = int(diff[(x, y, z)] or 0)
        counts[terrain_id] = counts.get(terrain_id, 0) + 1

  0%|          | 0/262144 [00:00<?, ?it/s]

In [146]:
records = []
for placer_id, counts in placer_counts.items():
    for terrain_id, count in counts.items():
        records.append([placer_id, int(terrain_id or 0), count])

In [147]:
df = pd.DataFrame.from_records(
    data=records,
    columns=["user_id", "terrain_id", "count"],
)

In [153]:
df.sort_values(by=["user_id", "count", "terrain_id"], ascending=[True, False, True])

,user_id,terrain_id,count
6267,7580077462299,2,1
6266,7580077462299,16777234,1
8895,10336469241985,2,1
8896,10336469241985,16777234,1
8324,11025567183527,77,1
...,...,...,...
5580,8992728197884243,5,1
10404,8996862785562361,2,1
10405,8996862785562361,16777268,1
10218,8996862785562364,2,1


In [157]:
df[["terrain_id", "count"]].groupby(by="terrain_id").sum().sort_values(by="count", ascending=False)

,count
terrain_id,
16777217,122505
2,55565
4,34309
1,30186
3,27666
...,...
84,8
16777277,2
16777270,1


In [170]:
top_users = df[["user_id", "count"]].groupby(by="user_id").sum().sort_values(by="count", ascending=False)
top_users.head(30)

,count
user_id,
5518985451091899,120528
8567554765071677,94163
7619355991105671,36583
2516585699511485,21994
2301587140303868,19651
5692638133546671,16731
8521385202672298,16375
8277444529729840,16211
8521385202672283,15315


In [169]:
top_users.where(top_users["count"] >= 500).dropna()

,count
user_id,
5518985451091899,120528.0
8567554765071677,94163.0
7619355991105671,36583.0
2516585699511485,21994.0
2301587140303868,19651.0
...,...
6316960872823341,509.0
4554937424429561,509.0
274260982586409,507.0
